# 08 - SharePoint: remote retrieval without an indexer

## Required permissions

- **You:** **Search Service Contributor** + **Search Index Data Reader** (or **Search Index Data Contributor**) on Search, `az login` in your tenant, and access to the SharePoint site/documents you should retrieve.
- **Microsoft 365:** each querying user needs a **Microsoft 365 Copilot add-on license**, or the tenant must explicitly enable Copilot Retrieval API pay-as-you-go for that user. A GitHub Copilot license is not sufficient.
- **Existing resources and test content:** an Azure AI Search service and a dedicated SharePoint test site with non-sensitive documents. Confirm licensing or billing eligibility before querying. No ingestion app, app admin consent, indexer or tenant administrator role is needed to run this remote path.

**Resource prefix:** set `WORKSHOP_PREFIX`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). This lab creates `alice01-sp-remote-source` and `alice01-sp-remote-kb`; it does not rename the existing site or copy its documents into an index. Prefixes prevent naming collisions, not enforce access isolation; retrieval still uses each user's Microsoft 365 permissions.

Query SharePoint through a **remote SharePoint knowledge source** and a knowledge base, using direct REST calls and no Vision skills. Search stores no copy of the content: no index, indexer or skillset, and no custom app registration.

| What runs | Where content lives | App registration |
|---|---|---|
| Remote SharePoint knowledge source -> knowledge base retrieve -> Copilot Retrieval API | SharePoint | None |

Retrieval evaluates the **signed-in user's** permissions in Microsoft 365. It is not a replacement for testing your source's permission model. Notebook 07 covers the indexed alternative; section 3 here compares the two.

**Run order:** run sections 1-2. Cleanup is opt-in.

**C# track:** .NET 10 SDK, Azure CLI and a C#/.NET Interactive notebook kernel
(Polyglot Notebooks). All helpers are inline below; no external source files,
SDK packages, model dependencies, local cache or prompts are required.
Sign in as a user, not a service principal or managed identity.


## Prerequisites and safe test content

- Use your tenant and a dedicated **non-sensitive test SharePoint site**, not a production library. Sign in with `az login --tenant <AZURE_TENANT_ID>` using the tenant ID from your settings.
- Your user needs Search Service Contributor (to create the knowledge source/base) and Search Index Data Reader or Contributor on the existing Search service.
- Remote queries require a Microsoft 365 Copilot add-on license for **each querying user**, or enabled Retrieval API pay-as-you-go billing for that user. Pay-as-you-go enablement requires at least one Copilot license in the tenant. It is separate from Search/Foundry billing.
- Upload the synthetic documents in `data/search` to the test site's default Documents library as Word documents: a general onboarding document and a restricted travel-policy document. Avoid real employee or customer data.
- Add `SHAREPOINT_SITE_URL` and a unique `WORKSHOP_PREFIX` to your untracked `.env`. Never paste access tokens or client secrets into cells, output, or chat.

These are preview APIs. Review the linked documentation before running.

## 1. Configure your lab

Set `AZURE_TENANT_ID`, `AZURE_SEARCH_ENDPOINT`, `SHAREPOINT_SITE_URL` and
`WORKSHOP_PREFIX` in the repository-root `.env` and `.env.azure`. Run the C#
kernel from the **c-sharp-notebooks** directory so `Path.GetFullPath("..")`
resolves the settings root. `.env` loads first, then `.env.azure`; file values
override environment variables and `.env.azure` wins.

The inline BCL helpers below validate a tenant GUID, a required 3–20 lowercase
letter/digit prefix starting with a letter, a public Azure Search HTTPS service
root and a safe public SharePoint HTTPS site URL before authentication/writes.
Use the tenant root, `/sites/name` or `/teams/name`, not a document, sharing link,
query or fragment. This safety check is stricter than Python08's general URL
check. No infrastructure detection or notebook 07 prerequisite.

Azure CLI supplies a **signed-in USER** Search-audience token. Retrieval sends
the same user identity in the raw `x-ms-query-source-authorization` header.
The inline transport prints error status, UTC Date, request/correlation headers
and a bounded response body before raising. It never turns a gateway failure,
HTTP 206 or source activity error into an empty success.


In [1]:
using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Net;
using System.Net.Http;
using System.Threading;
using System.Threading.Tasks;
using System.Diagnostics;
using System.Net.Http.Headers;
using System.Text;
using System.Text.Json.Nodes;
using System.Text.RegularExpressions;

public sealed record RemoteSettings(string Tenant, string SearchEndpoint, string SiteUrl, string Prefix)
{
    public string SourceName => Prefix + "-sp-remote-source";
    public string BaseName => Prefix + "-sp-remote-kb";

    public static RemoteSettings Load(string root)
    {
        LoadEnvFile(Path.Combine(root, ".env"));
        LoadEnvFile(Path.Combine(root, ".env.azure"));
        string Required(string name) => Environment.GetEnvironmentVariable(name)?.Trim() is { Length: > 0 } value
            ? value : throw new InvalidOperationException("Missing setting: " + name);
        var settings = new RemoteSettings(Required("AZURE_TENANT_ID"),
            Required("AZURE_SEARCH_ENDPOINT").TrimEnd('/'), Required("SHAREPOINT_SITE_URL").TrimEnd('/'),
            Required("WORKSHOP_PREFIX"));
        settings.Validate();
        return settings;
    }

    public void Validate()
    {
        if (!Guid.TryParseExact(Tenant, "D", out _) || !Regex.IsMatch(Prefix, @"^[a-z][a-z0-9]{2,19}$"))
            throw new InvalidOperationException("Use a tenant GUID and a 3-20 lowercase letter/digit prefix starting with a letter.");
        if (!SafeHttps(SearchEndpoint, out var search) ||
            !Regex.IsMatch(search!.Host, @"^[a-z0-9][a-z0-9-]*\.search\.windows\.net$", RegexOptions.IgnoreCase) ||
            search.AbsolutePath != "/")
            throw new InvalidOperationException("Use a public Azure HTTPS Search service root, without a path/query.");
        if (!SafeHttps(SiteUrl, out var site) ||
            !Regex.IsMatch(site!.Host, @"^[a-z0-9][a-z0-9-]*\.sharepoint\.com$", RegexOptions.IgnoreCase) ||
            !Regex.IsMatch(Uri.UnescapeDataString(site.AbsolutePath).TrimEnd('/'),
                @"^$|^/(sites|teams)/[a-zA-Z0-9 _.-]+$", RegexOptions.IgnoreCase) ||
            Uri.UnescapeDataString(site.AbsolutePath).Split('/').Any(part => part is "." or ".."))
            throw new InvalidOperationException("Use a public-cloud SharePoint HTTPS site root or /sites/name or /teams/name, not a file or sharing-link URL.");
    }

    static bool SafeHttps(string value, out Uri? uri)
    {
        uri = null;
        if (value.Any(c => char.IsControl(c) || c is ';' or '"' or '\\') ||
            Uri.UnescapeDataString(value).Any(c => char.IsControl(c) || c is ';' or '"' or '\\'))
            return false;
        return Uri.TryCreate(value, UriKind.Absolute, out uri) && uri.Scheme == "https" &&
            uri.UserInfo == "" && uri.IsDefaultPort && uri.Query == "" && uri.Fragment == "";
    }

    public static void LoadEnvFile(string path)
    {
        if (!File.Exists(path)) return;
        foreach (string line in File.ReadAllLines(path))
        {
            string setting = line.Trim();
            if (setting.Length == 0 || setting.StartsWith('#')) continue;
            if (setting.StartsWith("export ", StringComparison.Ordinal)) setting = setting[7..].TrimStart();
            int equals = setting.IndexOf('=');
            if (equals <= 0) continue;
            string name = setting[..equals].Trim();
            string value = setting[(equals + 1)..].Trim();
            if (value.Length >= 2 && ((value[0] == '"' && value[^1] == '"') || (value[0] == '\'' && value[^1] == '\'')))
                value = value[1..^1];
            else
            {
                int comment = value.IndexOf(" #", StringComparison.Ordinal);
                if (comment >= 0) value = value[..comment].TrimEnd();
            }
            Environment.SetEnvironmentVariable(name, value);
        }
    }
}

public sealed class RemoteTransport(HttpClient client, string tenant, Func<Task<string>>? tokenProvider = null)
{
    public const string Audience = "https://search.azure.com/";
    public const string ApiVersion = "2026-08-01-preview";
    readonly Func<Task<string>> getToken = tokenProvider ?? (() => AzureCliUserTokenAsync(tenant));

    public async Task<JsonObject> SendAsync(RemoteSettings settings, HttpMethod method, string path,
        JsonObject? body = null, bool queryUser = false, bool allowMissing = false)
    {
        settings.Validate();
        using var request = new HttpRequestMessage(method, $"{settings.SearchEndpoint}/{path}?api-version={ApiVersion}");
        request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await getToken());
        // Python08 sends the raw user token here, NOT another "Bearer " prefix.
        if (queryUser) request.Headers.Add("x-ms-query-source-authorization", await getToken());
        if (body is not null)
            request.Content = new StringContent(body.ToJsonString(), Encoding.UTF8, "application/json");
        using var response = await client.SendAsync(request);
        string text = await response.Content.ReadAsStringAsync();
        int status = (int)response.StatusCode;
        if (allowMissing && status == 404) return new JsonObject();
        if (!response.IsSuccessStatusCode)
        {
            Console.Error.WriteLine($"Remote request failed. HTTP: {status} {response.ReasonPhrase}");
            Console.Error.WriteLine("UTC (Date): " + response.Headers.Date?.ToString("R"));
            foreach (var header in response.Headers)
                if (header.Key.Contains("request", StringComparison.OrdinalIgnoreCase) ||
                    header.Key.Contains("correlation", StringComparison.OrdinalIgnoreCase) ||
                    header.Key.Contains("trace", StringComparison.OrdinalIgnoreCase) ||
                    header.Key.Equals("retry-after", StringComparison.OrdinalIgnoreCase))
                    Console.Error.WriteLine($"{header.Key}: {string.Join(", ", header.Value)}");
            Console.Error.WriteLine("Response body: " + text[..Math.Min(8000, text.Length)]);
            throw new HttpRequestException($"HTTP {status}: remote request failed; inspect UTC/correlation details and response body above.", null, response.StatusCode);
        }
        JsonObject result = string.IsNullOrWhiteSpace(text) && method != HttpMethod.Post
            ? new JsonObject() : JsonNode.Parse(text)!.AsObject();
        if (method == HttpMethod.Post)
        {
            Console.WriteLine(result.ToJsonString(new System.Text.Json.JsonSerializerOptions { WriteIndented = true }));
            if (status == 206 || result["activity"] is JsonArray activity && activity.Any(item => item?["error"] is not null))
                throw new InvalidOperationException("Retrieval was partial or a source failed. Inspect the activity output above.");
            RemoteResults.VerifyUrls(result, settings.SiteUrl);
            if (result["references"] is not JsonArray references || references.Count == 0)
                Console.WriteLine("No source references returned. Check query, source permissions, M365 indexing and eligibility.");
        }
        return result;
    }

    public static async Task<string> AzureCliUserTokenAsync(string tenant)
    {
        if (!Guid.TryParseExact(tenant, "D", out _)) throw new InvalidOperationException("Invalid tenant GUID.");
        JsonObject account = JsonNode.Parse(await RunCliAsync(["account", "show", "--output", "json"]))!.AsObject();
        if (account["user"]?["type"]?.GetValue<string>() != "user" ||
            !string.Equals(account["tenantId"]?.GetValue<string>(), tenant, StringComparison.OrdinalIgnoreCase))
            throw new InvalidOperationException("Sign in as a USER with az login --tenant AZURE_TENANT_ID; service principals/managed identities are not this lab's query identity.");
        JsonObject token = JsonNode.Parse(await RunCliAsync(
            ["account", "get-access-token", "--resource", Audience, "--tenant", tenant, "--output", "json"]))!.AsObject();
        return token["accessToken"]?.GetValue<string>() is { Length: > 0 } value
            ? value : throw new InvalidOperationException("Azure CLI returned no access token.");
    }

    static async Task<string> RunCliAsync(string[] arguments)
    {
        var start = new ProcessStartInfo
        {
            FileName = OperatingSystem.IsWindows() ? "cmd.exe" : "az",
            UseShellExecute = false,
            RedirectStandardOutput = true,
            RedirectStandardError = true,
            CreateNoWindow = true
        };
        if (OperatingSystem.IsWindows())
            foreach (string argument in new[] { "/d", "/c", "az" }) start.ArgumentList.Add(argument);
        foreach (string argument in arguments) start.ArgumentList.Add(argument);
        using var process = Process.Start(start) ?? throw new InvalidOperationException("Could not start Azure CLI.");
        Task<string> output = process.StandardOutput.ReadToEndAsync();
        Task<string> errors = process.StandardError.ReadToEndAsync();
        using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(60));
        try { await process.WaitForExitAsync(timeout.Token); }
        catch (OperationCanceledException)
        {
            if (!process.HasExited) process.Kill(entireProcessTree: true);
            await process.WaitForExitAsync();
            await Task.WhenAll(output, errors);
            throw new InvalidOperationException("Azure CLI authentication timed out. Sign in to your tenant.");
        }
        await errors;
        if (process.ExitCode != 0)
            throw new InvalidOperationException("Azure CLI authentication failed. Run az login --tenant with AZURE_TENANT_ID.");
        return await output;
    }
}

public static class RemoteResults
{
    public static void VerifyUrls(JsonNode? node, string siteUrl)
    {
        if (node is JsonObject obj)
        {
            foreach (var property in obj)
            {
                if (property.Key.Equals("webUrl", StringComparison.OrdinalIgnoreCase) && property.Value is not null)
                {
                    string url = property.Value.GetValue<string>();
                    var site = new Uri(siteUrl);
                    string path = Uri.UnescapeDataString(site.AbsolutePath).TrimEnd('/');
                    bool allowed = Uri.TryCreate(url, UriKind.Absolute, out var result) &&
                        result.Scheme == "https" && result.UserInfo == "" && result.IsDefaultPort &&
                        result.Host.Equals(site.Host, StringComparison.OrdinalIgnoreCase) &&
                        (Uri.UnescapeDataString(result.AbsolutePath).Equals(path, StringComparison.OrdinalIgnoreCase) ||
                         Uri.UnescapeDataString(result.AbsolutePath).StartsWith(path + "/", StringComparison.OrdinalIgnoreCase));
                    Console.WriteLine((allowed ? "Verified site webUrl: " : "WARNING: webUrl outside selected site: ") + url);
                    if (!allowed)
                        throw new InvalidOperationException("Returned webUrl is outside the selected site. Path KQL is not an authorization boundary; inspect the output.");
                }
                else VerifyUrls(property.Value, siteUrl);
            }
        }
        else if (node is JsonArray array)
            foreach (JsonNode? item in array) VerifyUrls(item, siteUrl);
    }
}


// Run the C# kernel from c-sharp-notebooks: load parent .env, then .env.azure.
RemoteSettings settings = RemoteSettings.Load(Path.GetFullPath(".."));
HttpClient client = new HttpClient { Timeout = TimeSpan.FromSeconds(120) };
Func<Task<string>> tokenProvider = () => RemoteTransport.AzureCliUserTokenAsync(settings.Tenant);
RemoteTransport Api() => new RemoteTransport(client, settings.Tenant, tokenProvider);
Console.WriteLine("Search: " + settings.SearchEndpoint);
Console.WriteLine("SharePoint site: " + settings.SiteUrl);
Console.WriteLine("API version: " + RemoteTransport.ApiVersion);


Search: https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net
SharePoint site: https://mngenvmcap272547.sharepoint.com/sites/dataforfishing
API version: 2026-08-01-preview


## 2. Create the remote knowledge source, knowledge base and query

No ingestion app, permissions or federated credential are required. This path creates only a remote knowledge source and knowledge base in Search. It invokes the Copilot Retrieval API with the querying user's identity; Search does not run Text Split or copy documents into an index.

We use **minimal reasoning effort + extractive data**: no LLM query planning or answer synthesis, and no Foundry model/RBAC changes are needed. A `semantic` intent passes the query directly to retrieval. This API term does not create an embedding model or Vision skill.

The `Path` KQL filter selects the test site. **It is not an authorization boundary**: invalid KQL can be ignored by the upstream Retrieval API, so inspect returned `webUrl` values. Source permissions remain the actual boundary. Results depend on Microsoft 365 indexing, supported file formats and license/billing eligibility.

**Gateway errors:** the query cell prints the HTTP status, UTC date, request ID and error body before raising. A message such as `Copilot retrieval call failed ... BadGateway ... /beta/copilot/retrieval` identifies a downstream Microsoft 365 Retrieval API failure, not an empty result or a missing Content Understanding role. Verify the user's Copilot license or pay-as-you-go eligibility and Microsoft 365 service health. If a manual retry still fails, retain the correlation details for support; do not change app permissions or enable billing merely because a gateway error occurred.

Run the following three cells in order. Each cell performs only its named REST
action: source PUT, base PUT, then one **billable/eligibility-gated** retrieve
POST. Rerunning retrieval never writes prerequisites. Successful results are
printed as extractive evidence, not an LLM answer. The helper checks every
returned `webUrl` it encounters against the selected site's host/path and fails
on out-of-site URLs; still inspect references/source data yourself.


### Create only the remote knowledge source

PUT `/knowledgesources/{name}` with the visible Path KQL and metadata payload.

In [ ]:
JsonObject remote_source = new JsonObject
{
    ["name"] = settings.SourceName,
    ["kind"] = "remoteSharePoint",
    ["description"] = "Workshop SharePoint retrieval without copying content into Search.",
    ["remoteSharePointParameters"] = new JsonObject
    {
        ["filterExpression"] = $"Path:\"{settings.SiteUrl}\"",
        ["resourceMetadata"] = new JsonArray("Author", "Title")
    }
};
await Api().SendAsync(settings, HttpMethod.Put, $"knowledgesources/{settings.SourceName}", remote_source);
Console.WriteLine("Remote knowledge source ready: " + settings.SourceName);


### Create only the extractive knowledge base

PUT `/knowledgebases/{name}`. The source must already exist; this cell does not create it or retrieve data.

In [ ]:
JsonObject remote_base = new JsonObject
{
    ["name"] = settings.BaseName,
    ["description"] = "Retrieve SharePoint passages without LLM planning or synthesis.",
    ["knowledgeSources"] = new JsonArray(new JsonObject { ["name"] = settings.SourceName }),
    ["models"] = new JsonArray(),
    ["retrievalReasoningEffort"] = new JsonObject { ["kind"] = "minimal" },
    ["outputMode"] = "extractiveData"
};
await Api().SendAsync(settings, HttpMethod.Put, $"knowledgebases/{settings.BaseName}", remote_base);
Console.WriteLine("Extractive knowledge base ready: " + settings.BaseName);


### Retrieve as your signed-in user

POST `/knowledgebases/{name}/retrieve` once. Edit `question` below. Confirm each user's license or enabled pay-as-you-go eligibility **before** running.

In [ ]:
string question = "travel reimbursement receipts";
if (string.IsNullOrWhiteSpace(question) || question.EnumerateRunes().Count() > 1500)
    throw new ArgumentException("Use a non-empty question of at most 1,500 characters.");
JsonObject retrieve_payload = new JsonObject
{
    ["intents"] = new JsonArray(new JsonObject { ["type"] = "semantic", ["search"] = question }),
    ["knowledgeSourceParams"] = new JsonArray(new JsonObject
    {
        ["knowledgeSourceName"] = settings.SourceName, ["kind"] = "remoteSharePoint",
        ["includeReferences"] = true, ["includeReferenceSourceData"] = true
    })
};
JsonObject remote_result = new JsonObject(); // Never retain a previous successful result after failure.
// queryUser sends Authorization: Bearer <user token> and x-ms-query-source-authorization: <raw user token>.
// SendAsync prints status/UTC/correlation/body before failing; no automatic paid retries.
remote_result = await Api().SendAsync(settings, HttpMethod.Post,
    $"knowledgebases/{settings.BaseName}/retrieve", retrieve_payload, queryUser: true);


### Inspect the knowledge base in the portal

You can also inspect your existing `<prefix>-sp-remote-kb` in the Azure AI Search portal. The **Knowledge sources** list shows the attached remote SharePoint source; the example below displays retrieved text with a linked source document.

<img src="../docs/sharepoint-remote-knowledge-base-portal.png" alt="Azure AI Search remote SharePoint knowledge base showing an active knowledge source and retrieved text with a source-document reference" width="1200" />

The fishing query and document are illustrative, not required sample content or expected results. This notebook configures **extractive data**, without LLM query planning or answer synthesis. Inspect the source reference and **Debug** retrieval details rather than treating the chat-style display as proof of a generated answer or complete permission enforcement. Additional portal queries are live Retrieval API calls and require the same user access and licensing or billing eligibility.

## 3. Compare with the indexed path (notebook 07)

If you also ran notebook 07, run the same question under the same two identities on both paths. Compare indexed `sourceFile`/`webUrl`/`content` and page ranges with remote references, `webUrl`, source data and activity records. Retrieval can return different passages/rankings; there is no guarantee of equal result counts or ordering.

| Capability | Indexed | Remote |
|---|---|---|
| Content Understanding, text embeddings and Search skillsets | Yes, subject to ACL-preview restrictions | No Search indexer or skillset |
| Document copy in Search | Yes | No |
| Ingestion app + federation | Required for notebook 07's secretless ingestion | Not required |
| Permission freshness | Depends on ingestion runs and inherited-ACL resync | Microsoft 365 evaluates access; service propagation/indexing latency still applies |
| Multimodal extraction in this lab | Not enabled | Retrieval API path is textual; images/charts are not a multimodal retrieval pipeline |
| Additional eligibility | Indexer preview and tenant admin setup | Copilot license or enabled Retrieval API pay-as-you-go per user |

Remote limits include 200 requests/user/hour, 1,500 query characters and up to 25 results; verify current limits before querying. A newly uploaded document might not immediately appear in remote retrieval. A syntactically accepted knowledge source does not prove user licensing, source access or successful retrieval.

**Acceptance exercise:** demonstrate a known allowed document and a known denied document for both users. Use direct user/Entra group document access for the comparison with notebook 07; test native SharePoint-group access separately on the remote path. Do not call the permission exercise complete based only on a fluent answer, an empty response, or a successful resource PUT.

## 4. Optional Search-object cleanup

Enable the following cell to remove only this notebook's knowledge base and knowledge source. Missing objects (404) are explicitly allowed; authorization failures and other errors are not ignored. Nothing in SharePoint or Entra was created by this notebook, so there is nothing else to remove.

Clear all notebook outputs before sharing; outputs can contain private SharePoint document text and URLs even though tokens are not printed.

Objects remain in Search unless you explicitly set `CLEAN_UP = true`. Cleanup deletes the derived knowledge base first, then the remote source; no site, content, Entra app, index or shared service is deleted.

In [ ]:
bool CLEAN_UP = false;
if (CLEAN_UP)
{
    foreach (var (collection, name) in new[]
    {
        ("knowledgebases", settings.BaseName), ("knowledgesources", settings.SourceName)
    })
    {
        await Api().SendAsync(settings, HttpMethod.Delete, $"{collection}/{name}", allowMissing: true);
        Console.WriteLine("Deleted or already absent: " + name);
    }
}
else
    Console.WriteLine("Search-object cleanup disabled.");


## References

- [Remote SharePoint knowledge source and licensing](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-sharepoint-remote)
- [Create a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-create-knowledge-base)
- [Minimal reasoning effort](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-set-retrieval-reasoning-effort)
- [Retrieve action and semantic intents](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve)

Preview API versions and Microsoft 365 eligibility can change.